# Chapter 14 — Bring Back Only What You Need

## Question

**When external information becomes available again, does finding it mean it should enter the context?**

Falsifiable structure: with the candidate pool frozen, do different admission policies admit different token totals and different required-token coverage? If yes, generation and admission are separate decisions. With no model call, only retrieved and admitted are demonstrated; used and helpful stay unmeasured.

## Setup — fourteen artifacts, frozen ranking, hidden labels

Fixture labels (REQUIRED / HELPFUL / DISTRACTOR / IRRELEVANT / TRAP) belong to the evaluator. Policies receive only visible fields; `make_visible` strips the labels, and an assertion enforces the strip.

In [ ]:
ARTIFACTS = [
    {'id': 'trap-migration', 'tokens': 1200, 'rank': 1},
    {'id': 'req-config', 'tokens': 300, 'rank': 2},
    {'id': 'dist-big', 'tokens': 8000, 'rank': 3},
    {'id': 'help-log', 'tokens': 600, 'rank': 4},
    {'id': 'dist-a', 'tokens': 900, 'rank': 5},
    {'id': 'req-incident', 'tokens': 50, 'rank': 6},
    {'id': 'dist-b', 'tokens': 700, 'rank': 7},
    {'id': 'dist-c', 'tokens': 500, 'rank': 8},
    {'id': 'help-note', 'tokens': 400, 'rank': 9},
    {'id': 'irr-1', 'tokens': 150, 'rank': 10},
    {'id': 'irr-2', 'tokens': 120, 'rank': 11},
    {'id': 'irr-3', 'tokens': 200, 'rank': 12},
    {'id': 'irr-4', 'tokens': 100, 'rank': 13},
    {'id': 'help-extra', 'tokens': 350, 'rank': 14},
]
TRUTH = {'req-incident': 'REQUIRED', 'req-config': 'REQUIRED', 'help-log': 'HELPFUL',
         'help-note': 'HELPFUL', 'help-extra': 'HELPFUL', 'dist-big': 'DISTRACTOR',
         'dist-a': 'DISTRACTOR', 'dist-b': 'DISTRACTOR', 'dist-c': 'DISTRACTOR',
         'trap-migration': 'TRAP', 'irr-1': 'IRRELEVANT', 'irr-2': 'IRRELEVANT',
         'irr-3': 'IRRELEVANT', 'irr-4': 'IRRELEVANT'}
# Frozen candidate pool and ranking: generation is held constant for every policy below.
POOL = sorted([a['id'] for a in ARTIFACTS], key=lambda i: next(a['rank'] for a in ARTIFACTS if a['id'] == i))

def make_visible(artifact_id):
    a = next(x for x in ARTIFACTS if x['id'] == artifact_id)
    return {'identity': a['id'], 'source_artifact': a['id'], 'generator': 'frozen-fixture',
            'rank': a['rank'], 'representation': 'SECTION', 'token_cost': a['tokens'],
            'provenance': 'store', 'scope': 'project'}

VISIBLE = [make_visible(i) for i in POOL]
assert all('label' not in v and TRUTH[v['identity']] not in v.values() for v in VISIBLE)
print(f'{len(POOL)} candidates frozen; labels stripped from policy input.')

## Baseline — candidate precision hides token waste

One 50-token required artifact beside one 8,000-token distractor: moderate candidate precision, enormous token-level waste.

In [ ]:
req, big = 50, 8000
print(f'candidate precision: 1/2 = 50%; required token share: {req}/{req + big} = {req / (req + big):.1%}')
assert req / (req + big) < 0.01
print('Candidate counts can hide token-level over-admission.')

## Intervention — six admission policies over the frozen pool

A admit none; B preload all; C fixed top-5; D deterministic gate over visible fields only; E identifier-first disclosure; F oracle minimum (ceiling, forbidden knowledge logged).

In [ ]:
IDENTITY_TOKENS, ANCHOR_TOKENS = 20, 120

def gate_visible(v):
    """System-selected admission. Reads rank and token_cost only."""
    if v['rank'] == 1 and v['token_cost'] > 1000:
        return 'DEFER'
    if v['rank'] <= 3 and v['token_cost'] <= 1000:
        return 'ADMIT-section'
    if v['rank'] <= 6:
        return 'ADMIT-anchor'
    return 'REJECT'

admitted = {
    'A none': {},
    'B preload-all': {i: 'FULL' for i in POOL},
    'C top-5': {i: 'SECTION' for i in POOL[:5]},
    'D gate': {},
    'E ident-first': {i: ('ANCHOR' if next(a['rank'] for a in ARTIFACTS if a['id'] == i) <= 4 else 'IDENTITY') for i in POOL},
    'F oracle': {'req-incident': 'SECTION', 'req-config': 'SECTION'},
}
for v in VISIBLE:
    d = gate_visible(v)
    if d == 'ADMIT-section':
        admitted['D gate'][v['identity']] = 'SECTION'
    elif d == 'ADMIT-anchor':
        admitted['D gate'][v['identity']] = 'ANCHOR'
    elif d == 'DEFER':
        admitted['D gate'][v['identity']] = 'DEFERRED-identity-only'

def tokens_of(adm):
    total = 0
    for i, rep in adm.items():
        full = next(a['tokens'] for a in ARTIFACTS if a['id'] == i)
        total += {'FULL': full, 'SECTION': full, 'ANCHOR': ANCHOR_TOKENS,
                  'IDENTITY': IDENTITY_TOKENS, 'DEFERRED-identity-only': IDENTITY_TOKENS}[rep]
    return total

def coverage(adm):
    got = sum(next(a['tokens'] for a in ARTIFACTS if a['id'] == i)
              for i, rep in adm.items() if TRUTH[i] == 'REQUIRED' and rep in ('FULL', 'SECTION'))
    return got

print(f"{'policy':14s} {'n':>3s} {'tokens':>6s} {'req-cover':>9s} {'dist+irr':>8s}")
for name, adm in admitted.items():
    waste = sum(next(a['tokens'] for a in ARTIFACTS if a['id'] == i)
                for i, rep in adm.items() if TRUTH[i] in ('DISTRACTOR', 'IRRELEVANT') and rep in ('FULL', 'SECTION'))
    print(f'{name:14s} {len(adm):3d} {tokens_of(adm):6d} {coverage(adm):9d} {waste:8d}')
assert coverage(admitted['F oracle']) == 350
assert tokens_of(admitted['B preload-all']) > tokens_of(admitted['D gate'])
print('Retrieved information does not become context until admitted.')

## Progressive disclosure — identity, anchor, section, full

Expansion follows visible insufficiency only: the incident anchor names the deadlock; its section carries the exact lock order. The disclosure trap (sufficient-looking anchor, reversing detail) is expanded here by fixture script, not by policy cleverness.

In [ ]:
DISCLOSURE = [('IDENTITY', 20), ('ANCHOR', 120), ('SECTION', 50), ('FULL', 50)]
spent, path = 0, []
for level, cost in DISCLOSURE:
    spent = cost  # each step replaces, not stacks: only the richest admitted form is resident
    path.append(level)
print('disclosure path:', ' -> '.join(path), f'(resident at end: {spent} tokens)')
assert spent == 50
print('Full-fidelity section (50 tokens, 1/8 of its document) is not a whole-document summary:')
print('how detailed? (fidelity) is not how much of the source? (granularity)')

## Granularity arm — full, section, typed extraction

One required artifact (`req-config`, 300 tokens), three admission widths. Coverage identical; cost is not.

In [ ]:
granularity = {'FULL': 300, 'SECTION': 300, 'TYPED_EXTRACTION': 80}
for width, toks in granularity.items():
    print(f'{width:16s} {toks:4d} tokens, required covered: True')
assert granularity['TYPED_EXTRACTION'] < granularity['FULL']
print('Same coverage, different spend: granularity is an admission choice, not a fidelity claim.')


## Misses, stopping, wandering — three different failures

In [ ]:
def classify_failure(required_id, pool, adm):
    if required_id not in pool:
        return 'candidate_miss'
    if required_id not in adm:
        return 'admission_miss'
    return 'admitted'

print('required never pooled:      ', classify_failure('req-incident', [i for i in POOL if i != 'req-incident'], admitted['D gate']))
print('required pooled but gated out:', classify_failure('req-config', POOL, {'req-incident': 'SECTION'}))
assert classify_failure('req-incident', POOL, admitted['D gate']) == 'admitted'

# Premature stopping: anchor read as sufficient, reversing detail never admitted.
premature = {'trap-migration': 'ANCHOR'}
print('premature (anchor-only over reversing detail):', 'admission_miss inside progressive disclosure')
# Wandering: required evidence present after B, yet C and D expanded anyway.
wasted_expansions = ['C', 'D']
print(f'wandering: answer present after B; wasted expansions: {len(wasted_expansions)} (latency and calls, not behaviour)')

## Try it

1. Raise top-k to 8 and watch required coverage stay flat while distractor tokens jump.
2. Move `req-incident` to rank 12: the gate now rejects it — reclassify the failure with `classify_failure`.
3. Expand `dist-big` to FULL under policy E and confirm token waste dominates every other column.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(tokens_of({'dist-big': 'FULL'}))

## What this demonstrates

- Candidate generation and context admission are different decisions: one frozen pool, six different admitted bundles.
- Candidate counts can hide token-level over-admission (50% precision beside 99% waste).
- Retrieved information does not become context until admitted; candidate miss and admission miss are different failures.

## What this does not demonstrate

- That retrieved material was used, or that admitted material improved behaviour (unmeasured here).
- That top-k is universally bad, or that progressive disclosure is always cheaper.
- That the deterministic gate transfers to real retrieval, or that similarity equals relevance.
- That the oracle minimum is deployable.

## Connection to the chapter

So far candidates have come from outside the agent:

> So far candidates have come from outside the agent. An acting agent changes that: every plan, hypothesis, report, and summary it writes can become future context.

That is Chapter 15.